In [1]:
import os
import random
import numpy as np
import torch

SEED = 42

random.seed(SEED) # Python
np.random.seed(SEED) # NumPy
torch.manual_seed(SEED) # PyTorch
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# deterministic)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
os.environ["PYTHONHASHSEED"] = str(SEED)


# **Load data**

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd

# data = pd.read_csv('/content/drive/MyDrive/sara_sms112/SequenceFeaturesData_bact.csv') #bacterial data with seqeunce features only
#data = pd.read_csv('/content/drive/MyDrive/sara_sms112/NetworkFeaturesDataBucket_bact.csv') # bacterisl data with network + seqeunce features

data =  pd.read_csv('/content/drive/MyDrive/sara_sms112/FinalDataEukXcodonw.csv') # eukaryotic data with seqeunce features only
#data = pd.read_csv('/content/drive/MyDrive/sara_sms112/FinalDataEukXcodonwXstring_buckets.csv') #eukaryotic data with codonw + string features

In [4]:
# Nc column is of type 'object', make it numeric

data['Nc'] = pd.to_numeric(data['Nc'], errors='coerce')
data = data.dropna(subset=['Nc'])

# **CV-Baseline models**

In [ ]:
# Dropping textual columns, uncomment desired dataset

#data_cleaned = data.drop(columns=['patric_id', 'strand', 'SequenceNA', 'SequenceAA','Organism', 'Organism_Name', 'Gene_Description', 'length_NA']) # bacteria sequence features only
#data_cleaned = data.drop(columns=['patric_id', 'STRING_id', 'strand', 'SequenceNA', 'SequenceAA','Organism', 'Organism_Name', 'Gene_Description', 'length_NA', 'degree_centrality', 'betweenness_centrality', 'load_centrality', 'eigenvector_centrality', 'pagerank']) # for bacteria with network features
#data_cleaned = data.drop(columns=['ID', 'GeneID', 'Orientation', 'Protein_description', 'SequenceNA', 'SequenceAA','Organism', 'length_NA']) # Eukaryotic seqeunce only
data_cleaned = data.drop(columns=['ID', 'GeneID', 'Orientation', 'STRING', 'Protein_description', 'SequenceNA', 'SequenceAA', 'Organism', 'length_NA', 'degree_centrality', 'betweenness_centrality', 'load_centrality', 'eigenvector_centrality', 'pagerank']) # eukaryotic with network features

data_cleaned = data_cleaned.dropna()



In [ ]:
# ONLY run for network features; one-hot encoding categorical bucket columns

cat_cols = [
    'degree_centrality_bucket',
    'betweenness_centrality_bucket',
    'load_centrality_bucket',
    'eigenvector_centrality_bucket',
    'pagerank_bucket'
]

data_cleaned = pd.get_dummies(
    data_cleaned,
    columns=cat_cols,
    drop_first=False
)

bool_cols = data_cleaned.select_dtypes(include='bool').columns
data_cleaned[bool_cols] = data_cleaned[bool_cols].astype(int)

In [ ]:
from sklearn.utils import resample

df_majority = data_cleaned[data_cleaned['Essential'] == 0]
df_minority = data_cleaned[data_cleaned['Essential'] == 1]

df_majority_downsampled = resample(df_majority,
                                   replace=False,    # sample without replacement
                                   n_samples=len(df_minority),    # match minority class size
                                   random_state=SEED)

# Combine minority class with downsampled majority class
data_balanced = pd.concat([df_majority_downsampled, df_minority])

# Shuffle the balanced dataset
data_balanced = data_balanced.sample(frac=1, random_state=SEED).reset_index(drop=True)

logistic regression

In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, classification_report)

# 1. Split features/target (assuming `data_balanced` exists):
X = data_balanced.drop(columns=['Essential'])
y = data_balanced['Essential']

# 2. Train/test split (you can also forego this if you only care about CV scores)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

# 3. Build a pipeline: scale → logistic regression
pipeline = Pipeline([
    ("scaler", StandardScaler()),            # feature scaling
    ("clf", LogisticRegression(solver="liblinear", max_iter=1000, random_state=SEED))
])

# 4. Define the hyperparameter grid to search
param_grid = {
    "clf__C": [10, 100],        # inverse regularization strength
    "clf__penalty": ["l2"],            # L1 vs L2 regularization
    "clf__tol": [1e-4, 1e-5],
    "clf__max_iter": [100, 500, 1000]
}

# 5. Set up GridSearchCV with 5-fold CV
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    scoring="accuracy",       # you can also try 'f1', 'roc_auc', etc.
    verbose=3,
    n_jobs=1   # use all cores (try chaning to 1)
)

# 6. Run the grid search on the training set
grid_search.fit(X_train, y_train)

# 7. Best params and CV score
cv_results = grid_search.cv_results_
mean = grid_search.best_score_
std = cv_results['std_test_score'][grid_search.best_index_]
print("Best parameters:", grid_search.best_params_)
print(f"Best CV accuracy: {mean:.4f} ± {std:.4f}")

# 8. Evaluate on held-out test set
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]  # needed for ROC-AUC

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)      # same as sensitivity
f1 = f1_score(y_test, y_pred)
roc = roc_auc_score(y_test, y_proba)

print("\nTest Set Performance:")
print(f"Accuracy     : {acc:.4f}")
print(f"Precision    : {prec:.4f}")
print(f"Recall/Sens. : {rec:.4f}")
print(f"F1 Score     : {f1:.4f}")
print(f"ROC-AUC      : {roc:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Fitting 5 folds for each of 1 candidates, totalling 5 fits
[CV 1/5] END clf__C=100, clf__max_iter=100, clf__penalty=l2, clf__tol=1e-05;, score=0.771 total time=   1.6s
[CV 2/5] END clf__C=100, clf__max_iter=100, clf__penalty=l2, clf__tol=1e-05;, score=0.765 total time=   1.6s
[CV 3/5] END clf__C=100, clf__max_iter=100, clf__penalty=l2, clf__tol=1e-05;, score=0.753 total time=   1.7s
[CV 4/5] END clf__C=100, clf__max_iter=100, clf__penalty=l2, clf__tol=1e-05;, score=0.763 total time=   1.5s
[CV 5/5] END clf__C=100, clf__max_iter=100, clf__penalty=l2, clf__tol=1e-05;, score=0.774 total time=   1.3s
Best parameters: {'clf__C': 100, 'clf__max_iter': 100, 'clf__penalty': 'l2', 'clf__tol': 1e-05}
Best CV accuracy: 0.7651 ± 0.0073

Test Set Performance:
Accuracy     : 0.7556
Precision    : 0.7665
Recall/Sens. : 0.7349
F1 Score     : 0.7504
ROC-AUC      : 0.8442

Classification Report:
              precision    recall  f1-score   support

           0       0.75      0.78      0.76      1408


Random forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, classification_report)

# 1. Split features/target
X = data_balanced.drop(columns=['Essential'])
y = data_balanced['Essential']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

# 2. Pipeline (scaler not needed for RF, but pipeline keeps it consistent)
pipeline_rf = Pipeline([
    ("clf", RandomForestClassifier(random_state=SEED))
])

# 3. Hyperparameter grid
param_grid_rf = {
    "clf__n_estimators":      [200, 500],
    "clf__max_depth":         [20, 30],
    "clf__min_samples_split": [2, 5, 10],
    "clf__min_samples_leaf":  [1, 2, 4],
    "clf__max_features":      ["sqrt"]
}

# 4. GridSearchCV with 5-fold CV
grid_search_rf = GridSearchCV(
    estimator=pipeline_rf,
    param_grid=param_grid_rf,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    scoring="accuracy",   # still optimizes for accuracy
    verbose=3,
    n_jobs=-1             # -1 to use all cores
)

# 5. Run the search
grid_search_rf.fit(X_train, y_train)

# 6. Best params and CV score (mean ± std)
cv_results_rf = grid_search_rf.cv_results_
mean_rf = grid_search_rf.best_score_
std_rf = cv_results_rf['std_test_score'][grid_search_rf.best_index_]
print("Best RF params:", grid_search_rf.best_params_)
print(f"Best CV accuracy: {mean_rf:.4f} ± {std_rf:.4f}")

# 7. Evaluate on hold-out test set
best_rf = grid_search_rf.best_estimator_
y_pred_rf = best_rf.predict(X_test)
y_proba_rf = best_rf.predict_proba(X_test)[:, 1]  # for ROC-AUC

acc = accuracy_score(y_test, y_pred_rf)
prec = precision_score(y_test, y_pred_rf)
rec = recall_score(y_test, y_pred_rf)      # same as sensitivity
f1 = f1_score(y_test, y_pred_rf)
roc = roc_auc_score(y_test, y_proba_rf)

print("\nTest Set Performance:")
print(f"Accuracy     : {acc:.4f}")
print(f"Precision    : {prec:.4f}")
print(f"Recall/Sens. : {rec:.4f}")
print(f"F1 Score     : {f1:.4f}")
print(f"ROC-AUC      : {roc:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))


Fitting 5 folds for each of 1 candidates, totalling 5 fits
Best RF params: {'clf__max_depth': 30, 'clf__max_features': 'sqrt', 'clf__min_samples_leaf': 2, 'clf__min_samples_split': 10, 'clf__n_estimators': 500}
Best CV accuracy: 0.7760 ± 0.0098

Test Set Performance:
Accuracy     : 0.7908
Precision    : 0.8007
Recall/Sens. : 0.7740
F1 Score     : 0.7871
ROC-AUC      : 0.8619

Classification Report:
              precision    recall  f1-score   support

           0       0.78      0.81      0.79      1408
           1       0.80      0.77      0.79      1407

    accuracy                           0.79      2815
   macro avg       0.79      0.79      0.79      2815
weighted avg       0.79      0.79      0.79      2815



XGBoost

In [ ]:
import xgboost as xgb
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report
)
from scipy.stats import randint, uniform

# 1. Prepare train/test split
X = data_balanced.drop(columns=['Essential'])
y = data_balanced['Essential']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

# 2. Build pipeline (scaling not needed for XGBoost)
pipeline_xgb = Pipeline([
    ("clf", xgb.XGBClassifier(
        use_label_encoder=False,
        eval_metric="logloss",
        random_state=SEED,
        device='cuda',
        tree_method="hist",        # for better reproducibility
        predictor="gpu_predictor"
    ))
])

# 3. Randomized hyperparameter distributions
param_dist_xgb = {
    "clf__n_estimators": randint(100, 500),
    "clf__max_depth": randint(3, 50),
    "clf__learning_rate": uniform(0.01, 0.3),
    "clf__subsample": uniform(0.6, 0.4),          # samples between 0.6–1.0
    "clf__colsample_bytree": uniform(0.6, 0.4),   # samples between 0.6–1.0
    "clf__gamma": randint(0, 6),
    "clf__reg_alpha": randint(0, 6),
    "clf__reg_lambda": randint(1, 100)
}

# 4. Randomized Search (faster than full grid)
random_search_xgb = RandomizedSearchCV(
    estimator=pipeline_xgb,
    param_distributions=param_dist_xgb,
    n_iter=50,             # number of random parameter combinations to try
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    scoring="accuracy",
    verbose=1,
    random_state=SEED,
    n_jobs=-1
)

# 5. Fit the model
random_search_xgb.fit(X_train, y_train)

# 6. Best params & CV mean ± std
cv_results_xgb = random_search_xgb.cv_results_
mean_xgb = random_search_xgb.best_score_
std_xgb = cv_results_xgb['std_test_score'][random_search_xgb.best_index_]
print("Best XGB params:", random_search_xgb.best_params_)
print(f"Best CV accuracy: {mean_xgb:.4f} ± {std_xgb:.4f}")

# 7. Evaluate on hold-out test set
best_xgb = random_search_xgb.best_estimator_
y_pred_xgb = best_xgb.predict(X_test)
y_proba_xgb = best_xgb.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred_xgb)
prec = precision_score(y_test, y_pred_xgb)
rec = recall_score(y_test, y_pred_xgb)
f1 = f1_score(y_test, y_pred_xgb)
roc = roc_auc_score(y_test, y_proba_xgb)

print("\nTest Set Performance:")
print(f"Accuracy     : {acc:.4f}")
print(f"Precision    : {prec:.4f}")
print(f"Recall/Sens. : {rec:.4f}")
print(f"F1 Score     : {f1:.4f}")
print(f"ROC-AUC      : {roc:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_xgb))


Fitting 5 folds for each of 50 candidates, totalling 250 fits


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [20:07:39] WARNING: /workspace/src/context.cc:53: No visible GPU is found, setting device to CPU.
  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [20:07:39] WARNING: /workspace/src/context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [20:07:39] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "predictor", "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Best XGB params: {'clf__colsample_bytree': np.float64(0.881207583558071), 'clf__gamma': 0, 'clf__learning_rate': np.float64(0.029467674132694466), 'clf__max_depth': 14, 'clf__n_estimators': 394, 'clf__reg_alpha': 1, 'clf__reg_lambda': 3, 'clf__subsample': np.float64(0.7203513239267079)}
Best CV accuracy: 0.7965 ± 0.0078

Test Set Performance:
Accuracy     : 0.8021
Precision    : 0.8275
Recall/Sens. : 0.7634
F1 Score     : 0.7942
ROC-AUC      : 0.8808

Classification Report:
              precision    recall  f1-score   support

           0       0.78      0.84      0.81      1678
           1       0.83      0.76      0.79      1678

    accuracy                           0.80      3356
   macro avg       0.80      0.80      0.80      3356
weighted avg       0.80      0.80      0.80      3356



TabNet

In [ ]:
pip install pytorch-tabnet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 1.7 MB/s eta 0:00:00


In [ ]:
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.metrics import (make_scorer, accuracy_score, precision_score,
                             recall_score, f1_score, roc_auc_score, classification_report)
from pytorch_tabnet.tab_model import TabNetClassifier
from scipy.stats import uniform, randint

# --- 1. Train/Test Split ---
X = data_balanced.drop(columns=['Essential']).values
y = data_balanced['Essential'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

# --- 2. Scorer ---
scorer = make_scorer(accuracy_score)

# --- 3. Parameter Distributions for RandomizedSearch ---
param_dist = {
    'optimizer_params': [{'lr': lr} for lr in [1e-2, 1e-3, 5e-4, 1e-4]],
    'gamma': uniform(1.0, 0.5),        # range: [1.0, 1.5]
    'n_steps': randint(3, 10),         # integer 3–9
    'n_d': randint(8, 64),             # number of decision steps
    'n_a': randint(8, 64),             # number of attention steps
    'lambda_sparse': uniform(1e-5, 1e-3)
}

# --- 4. Initialize TabNet (GPU Enabled) ---
tabnet_model = TabNetClassifier(
    optimizer_fn=torch.optim.Adam,
    mask_type='entmax',
    verbose=0,
    seed=SEED,
    device_name='cuda' if torch.cuda.is_available() else 'cpu'  # ✅ Auto GPU
)

# --- 5. Randomized SearchCV ---
n_iter = 20  # test 20 random combinations
cv_folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

random_search_tabnet = RandomizedSearchCV(
    estimator=tabnet_model,
    param_distributions=param_dist,
    n_iter=n_iter,
    cv=cv_folds,
    scoring=scorer,
    verbose=2,
    random_state=SEED,
    n_jobs=1  # ⚠️ keep =1; TabNet uses GPU internally, multi-job may conflict
)

print(f"🔍 Running TabNet RandomizedSearchCV with {n_iter} parameter combinations...")

random_search_tabnet.fit(X_train, y_train)

# --- 6. Cross-Validation Results ---
cv_results_tabnet = random_search_tabnet.cv_results_
mean_tabnet = random_search_tabnet.best_score_
std_tabnet = cv_results_tabnet['std_test_score'][random_search_tabnet.best_index_]
print("\nBest TabNet Params:", random_search_tabnet.best_params_)
print(f"Best CV accuracy: {mean_tabnet:.4f} ± {std_tabnet:.4f}")

# --- 7. Evaluate on Hold-Out Test Set ---
best_tabnet = random_search_tabnet.best_estimator_

y_pred_tabnet = best_tabnet.predict(X_test)
y_proba_tabnet = best_tabnet.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred_tabnet)
prec = precision_score(y_test, y_pred_tabnet)
rec = recall_score(y_test, y_pred_tabnet)
f1 = f1_score(y_test, y_pred_tabnet)
roc = roc_auc_score(y_test, y_proba_tabnet)

print("\n📊 Test Set Performance:")
print(f"Accuracy     : {acc:.4f}")
print(f"Precision    : {prec:.4f}")
print(f"Recall/Sens. : {rec:.4f}")
print(f"F1 Score     : {f1:.4f}")
print(f"ROC-AUC      : {roc:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_tabnet))


🔍 Running TabNet RandomizedSearchCV with 20 parameter combinations...
Fitting 5 folds for each of 20 candidates, totalling 100 fits


/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/abstract_model.py:687: UserWarning: No early stopping will be performed, last training weights will be used.
  warnings.warn(wrn_msg)


[CV] END gamma=1.1872700594236814, lambda_sparse=0.0009607143064099162, n_a=50, n_d=15, n_steps=7, optimizer_params={'lr': 0.01}; total time= 2.7min


/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/abstract_model.py:687: UserWarning: No early stopping will be performed, last training weights will be used.
  warnings.warn(wrn_msg)


[CV] END gamma=1.1872700594236814, lambda_sparse=0.0009607143064099162, n_a=50, n_d=15, n_steps=7, optimizer_params={'lr': 0.01}; total time= 2.6min


...

[CV] END gamma=1.2247253370691018, lambda_sparse=0.0001054101164904113, n_a=62, n_d=30, n_steps=8, optimizer_params={'lr': 0.0001}; total time= 3.5min


/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/abstract_model.py:687: UserWarning: No early stopping will be performed, last training weights will be used.
  warnings.warn(wrn_msg)


[CV] END gamma=1.2247253370691018, lambda_sparse=0.0001054101164904113, n_a=62, n_d=30, n_steps=8, optimizer_params={'lr': 0.0001}; total time= 3.6min


/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/abstract_model.py:687: UserWarning: No early stopping will be performed, last training weights will be used.
  warnings.warn(wrn_msg)



Best TabNet Params: {'gamma': np.float64(1.1872700594236814), 'lambda_sparse': np.float64(0.0009607143064099162), 'n_a': 50, 'n_d': 15, 'n_steps': 7, 'optimizer_params': {'lr': 0.01}}
Best CV accuracy: 0.7816 ± 0.0070

📊 Test Set Performance:
Accuracy     : 0.7935
Precision    : 0.7781
Recall/Sens. : 0.8212
F1 Score     : 0.7991
ROC-AUC      : 0.8752

Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.77      0.79      1678
           1       0.78      0.82      0.80      1678

    accuracy                           0.79      3356
   macro avg       0.79      0.79      0.79      3356
weighted avg       0.79      0.79      0.79      3356



# **CV-TF-IDF**

In [5]:
# drop textual columns except for gene annotations

#data_cleaned = data.drop(columns=['patric_id', 'strand', 'SequenceNA', 'SequenceAA','Organism', 'Organism_Name', 'length_NA']) # bacteria sequence features only
#data_cleaned = data.drop(columns=['patric_id', 'STRING_id', 'strand', 'SequenceNA', 'SequenceAA','Organism', 'Organism_Name', 'length_NA', 'degree_centrality', 'betweenness_centrality', 'load_centrality', 'eigenvector_centrality', 'pagerank']) # for bacteria with network features
data_cleaned = data.drop(columns=['ID', 'GeneID', 'Orientation', 'SequenceNA', 'SequenceAA','Organism', 'length_NA']) #Eukaryotic seqeunce only
#data_cleaned = data.drop(columns=['ID', 'GeneID', 'Orientation', 'STRING', 'SequenceNA', 'SequenceAA', 'Organism', 'length_NA', 'degree_centrality', 'betweenness_centrality', 'load_centrality', 'eigenvector_centrality', 'pagerank']) #eukaryotic with network features

data_cleaned = data_cleaned.dropna()


In [16]:
# One-hot encode categorical bucket columns

cat_cols = [
    'degree_centrality_bucket',
    'betweenness_centrality_bucket',
    'load_centrality_bucket',
    'eigenvector_centrality_bucket',
    'pagerank_bucket'
]

data_cleaned = pd.get_dummies(
    data_cleaned,
    columns=cat_cols,
    drop_first=False  # keep all categories
)

# Convert boolean columns to integers
bool_cols = data_cleaned.select_dtypes(include='bool').columns
data_cleaned[bool_cols] = data_cleaned[bool_cols].astype(int)

In [6]:
from sklearn.utils import resample

df_majority = data_cleaned[data_cleaned['Essential'] == 0]
df_minority = data_cleaned[data_cleaned['Essential'] == 1]

# Downsample majority class
df_majority_downsampled = resample(df_majority,
                                   replace=False,    # sample without replacement
                                   n_samples=len(df_minority),    # match minority class size
                                   random_state=SEED)

# Combine minority class with downsampled majority class
data_balanced = pd.concat([df_majority_downsampled, df_minority])

# Shuffle the balanced dataset
data_balanced = data_balanced.sample(frac=1, random_state=SEED).reset_index(drop=True)

Logistic regression

In [10]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, classification_report)

X = data_balanced.drop(columns=["Essential"])
y = data_balanced["Essential"]

text_col = 'Protein_description' # exchange with 'Gene_Description' if using bacterial datasets, 'Protein_description' if eukaryotic dataset
num_cols = X.columns.drop(text_col)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

preprocessor = ColumnTransformer(
    transformers=[
        ("text", TfidfVectorizer(
            max_features=5000,
            ngram_range=(1, 2),
            stop_words="english"
        ), text_col),
        ("num", StandardScaler(), num_cols)
    ]
)

clf = LogisticRegression(
    C=100,
    penalty="l2",
    tol=1e-5,
    #solver="liblinear",
    max_iter=100,
    random_state=SEED
)

pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("clf", clf)
])

cv_scores = cross_val_score(
    pipeline,
    X_train,
    y_train,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    scoring="accuracy",
    n_jobs=1
)

print(f"CV Accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc = roc_auc_score(y_test, y_proba)

print("\nTest Set Performance:")
print(f"Accuracy     : {acc:.4f}")
print(f"Precision    : {prec:.4f}")
print(f"Recall/Sens. : {rec:.4f}")
print(f"F1 Score     : {f1:.4f}")
print(f"ROC-AUC      : {roc:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _c

CV Accuracy: 0.7975 ± 0.0045

Test Set Performance:
Accuracy     : 0.8063
Precision    : 0.8115
Recall/Sens. : 0.7980
F1 Score     : 0.8047
ROC-AUC      : 0.8856

Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.81      0.81      1678
           1       0.81      0.80      0.80      1678

    accuracy                           0.81      3356
   macro avg       0.81      0.81      0.81      3356
weighted avg       0.81      0.81      0.81      3356



/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Random Forest

In [11]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, classification_report)

X = data_balanced.drop(columns=["Essential"])
y = data_balanced["Essential"]

text_col = 'Protein_description'
num_cols = X.columns.drop(text_col)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

preprocessor = ColumnTransformer(
    transformers=[
        ("text", TfidfVectorizer(
            max_features=500,   # keep this modest for RF
            ngram_range=(1, 2),
            stop_words="english"
        ), text_col),
        ("num", "passthrough", num_cols)
    ]
)

rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=30,
    max_features="sqrt",
    min_samples_leaf=2,
    min_samples_split=10,
    random_state=SEED,
    n_jobs=-1
)

pipeline_rf = Pipeline([
    ("preprocess", preprocessor),
    ("clf", rf)
])

cv_scores = cross_val_score(
    pipeline_rf,
    X_train,
    y_train,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    scoring="accuracy",
    n_jobs=1
)

print(f"CV Accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

pipeline_rf.fit(X_train, y_train)

y_pred = pipeline_rf.predict(X_test)
y_proba = pipeline_rf.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc = roc_auc_score(y_test, y_proba)

print("\nTest Set Performance:")
print(f"Accuracy     : {acc:.4f}")
print(f"Precision    : {prec:.4f}")
print(f"Recall/Sens. : {rec:.4f}")
print(f"F1 Score     : {f1:.4f}")
print(f"ROC-AUC      : {roc:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))



CV Accuracy: 0.7914 ± 0.0077

Test Set Performance:
Accuracy     : 0.8096
Precision    : 0.8625
Recall/Sens. : 0.7366
F1 Score     : 0.7946
ROC-AUC      : 0.8893

Classification Report:
              precision    recall  f1-score   support

           0       0.77      0.88      0.82      1678
           1       0.86      0.74      0.79      1678

    accuracy                           0.81      3356
   macro avg       0.82      0.81      0.81      3356
weighted avg       0.82      0.81      0.81      3356



XGBoost

In [12]:
import xgboost as xgb
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, classification_report)



X = data_balanced.drop(columns=['Essential'])
y = data_balanced['Essential']

text_col = 'Protein_description'
num_cols = X.columns.drop(text_col)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

preprocessor = ColumnTransformer(
    transformers=[
        ("text", TfidfVectorizer(
            max_features=200,
            ngram_range=(1, 2),
            stop_words="english"
        ), text_col),
        ("num", "passthrough", num_cols)
    ]
)


xgb_model = xgb.XGBClassifier(
    n_estimators=394,
    max_depth=14,
    learning_rate=0.029,
    reg_alpha=1,
    reg_lambda=3,
    subsample=0.72,
    colsample_bytree=0.88,
    use_label_encoder=False,
    eval_metric="logloss",
    random_state=SEED,
    tree_method="hist",
    predictor="cuda",
    n_jobs=-1
)


pipeline_xgb = Pipeline([
    ("preprocess", preprocessor),
    ("clf", xgb_model)
])


cv_scores = cross_val_score(
    pipeline_xgb,
    X_train,
    y_train,
    cv= StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    scoring="accuracy",
    n_jobs=-1
)

print(f"CV Accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")


pipeline_xgb.fit(X_train, y_train)


y_pred = pipeline_xgb.predict(X_test)
y_proba = pipeline_xgb.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc = roc_auc_score(y_test, y_proba)

print("\nTest Set Performance:")
print(f"Accuracy     : {acc:.4f}")
print(f"Precision    : {prec:.4f}")
print(f"Recall/Sens. : {rec:.4f}")
print(f"F1 Score     : {f1:.4f}")
print(f"ROC-AUC      : {roc:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))


CV Accuracy: 0.8104 ± 0.0073


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [08:04:25] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "predictor", "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



Test Set Performance:
Accuracy     : 0.8179
Precision    : 0.8370
Recall/Sens. : 0.7896
F1 Score     : 0.8126
ROC-AUC      : 0.9023

Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.85      0.82      1678
           1       0.84      0.79      0.81      1678

    accuracy                           0.82      3356
   macro avg       0.82      0.82      0.82      3356
weighted avg       0.82      0.82      0.82      3356



TabNet

In [13]:
!pip install pytorch-tabnet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 3.3 MB/s eta 0:00:00


In [ ]:
from sklearn.model_selection import train_test_split, KFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report
from pytorch_tabnet.tab_model import TabNetClassifier

# -----------------------------
# 1️⃣ Prepare data
# -----------------------------
X = data_balanced.drop(columns=['Essential'])
y = data_balanced['Essential'].values

text_col = 'Protein_description'
num_cols = X.columns.drop(text_col)

X_train_df, X_test_df, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

# -----------------------------
# 2️⃣ TF-IDF + numerical preprocessing
# -----------------------------
tfidf = TfidfVectorizer(
    max_features=500,   # same as RF best
    ngram_range=(1,2),
    stop_words='english'
)

# Fit TF-IDF on train only
X_train_text = tfidf.fit_transform(X_train_df[text_col]).toarray()
X_test_text = tfidf.transform(X_test_df[text_col]).toarray()

# Get numerical features
X_train_num = X_train_df[num_cols].values
X_test_num = X_test_df[num_cols].values

# Combine numerical + TF-IDF
X_train = np.hstack([X_train_num, X_train_text])
X_test = np.hstack([X_test_num, X_test_text])

print(f"Final feature shape: Train {X_train.shape}, Test {X_test.shape}")

# -----------------------------
# 3️⃣ TabNet model with fixed parameters
# -----------------------------
tabnet_params = {
    'n_d': 15,
    'n_a': 50,
    'n_steps': 7,
    'gamma': 1.2,
    'lambda_sparse': 0,
    'optimizer_fn': torch.optim.Adam,
    'optimizer_params': {'lr': 0.01},
    'mask_type': 'entmax',
    'verbose': 0,
    'device_name': 'cuda' if torch.cuda.is_available() else 'cpu'
}

# -----------------------------
# 4️⃣ 5-fold Cross-Validation
# -----------------------------
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
cv_scores = []

for train_idx, val_idx in kf.split(X_train):
    X_tr, X_val = X_train[train_idx], X_train[val_idx]
    y_tr, y_val = y_train[train_idx], y_train[val_idx]

    model = TabNetClassifier(**tabnet_params)

    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        eval_metric=['accuracy'],
        max_epochs=100,
        patience=10,
        batch_size=256,
        virtual_batch_size=128,
        drop_last=False
    )

    y_val_pred = model.predict(X_val)
    acc = accuracy_score(y_val, y_val_pred)
    cv_scores.append(acc)

print(f"CV Accuracy: {np.mean(cv_scores):.4f} ± {np.std(cv_scores):.4f}")

# -----------------------------
# 5️⃣ Train on full training set
# -----------------------------
tabnet_full = TabNetClassifier(**tabnet_params)
tabnet_full.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    eval_metric=['accuracy'],
    max_epochs=100,
    patience=10,
    batch_size=256,
    virtual_batch_size=128,
    drop_last=False
)

# -----------------------------
# 6️⃣ Test set evaluation
# -----------------------------
y_pred = tabnet_full.predict(X_test)
y_proba = tabnet_full.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc = roc_auc_score(y_test, y_proba)

print("\n📊 Test Set Performance:")
print(f"Accuracy     : {acc:.4f}")
print(f"Precision    : {prec:.4f}")
print(f"Recall/Sens. : {rec:.4f}")
print(f"F1 Score     : {f1:.4f}")
print(f"ROC-AUC      : {roc:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Final feature shape: Train (13420, 557), Test (3356, 557)

Early stopping occurred at epoch 59 with best_epoch = 49 and best_val_0_accuracy = 0.78949


/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)



Early stopping occurred at epoch 44 with best_epoch = 34 and best_val_0_accuracy = 0.77012


/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


# **LOSO XGBoost**

loso Xgboost model trained on eukaryotic -no network features- dataset

In [ ]:
import pandas as pd
data =  pd.read_csv('/content/drive/MyDrive/sara_sms112/FinalDataEukXcodonw.csv') # eukaryotic data with seqeunce features only

In [ ]:
data['Organism'].unique()

In [ ]:
from sklearn.model_selection import train_test_split

# 1. Separate out Drosophila melanogaster
test_data = data[data["Organism"] == "Plasmodium falciparum"].copy()

# 2. Keep the rest (all other organisms)
train_eval_data = data[data["Organism"] != "Plasmodium falciparum"].copy()

In [ ]:
from sklearn.utils import resample

df_majority = train_eval_data[train_eval_data['Essential'] == 0]
df_minority = train_eval_data[train_eval_data['Essential'] == 1]

df_majority_downsampled = resample(df_majority,
                                   replace=False,    # sample without replacement
                                   n_samples=len(df_minority),    # match minority class size
                                   random_state=SEED)

# combine minority class with downsampled majority class
data_balanced = pd.concat([df_majority_downsampled, df_minority])

# shuffle the balanced dataset
data_balanced = data_balanced.sample(frac=1, random_state=SEED).reset_index(drop=True)

In [ ]:
data_cleaned = data_balanced.drop(columns=['ID', 'GeneID', 'SequenceNA', 'SequenceAA','Organism'])
data_cleaned = data_cleaned.dropna()

# for left-out-species
data_test_cleaned = test_data.drop(columns=['ID', 'GeneID', 'SequenceNA', 'SequenceAA','Organism'])
data_test_cleaned = data_test_cleaned.dropna()


In [ ]:
import xgboost as xgb
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report
)

# -----------------------------
# 1️⃣ Prepare LOSO data
# -----------------------------
X_train = data_cleaned.drop(columns=['Essential'])
y_train = data_cleaned['Essential']

X_test = data_test_cleaned.drop(columns=['Essential'])
y_test = data_test_cleaned['Essential']

text_col = "Protein_description"
num_cols = X_train.columns.drop(text_col)

# -----------------------------
# 2️⃣ Preprocessing
# -----------------------------
preprocessor = ColumnTransformer(
    transformers=[
        ("text", TfidfVectorizer(
            max_features=200,
            ngram_range=(1, 2),
            stop_words="english"
        ), text_col),
        ("num", "passthrough", num_cols)
    ]
)

X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

# -----------------------------
# 3️⃣ XGBoost model
# -----------------------------
xgb_model = xgb.XGBClassifier(
    n_estimators=400,
    max_depth=20,
    learning_rate=0.1,
    reg_alpha=0,
    reg_lambda=15,
    subsample=0.7,
    colsample_bytree=0.7,
    use_label_encoder=False,
    eval_metric="logloss",
    random_state=SEED,
    n_jobs=-1
)

# -----------------------------
# 4️⃣ Train
# -----------------------------
xgb_model.fit(X_train_transformed, y_train)

# -----------------------------
# 5️⃣ Evaluate on left-out species
# -----------------------------
y_pred = xgb_model.predict(X_test_transformed)
y_proba = xgb_model.predict_proba(X_test_transformed)[:, 1]

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc = roc_auc_score(y_test, y_proba)

print("\nLOSO Test Set Performance:")
print(f"Accuracy     : {acc:.4f}")
print(f"Precision    : {prec:.4f}")
print(f"Recall/Sens. : {rec:.4f}")
print(f"F1 Score     : {f1:.4f}")
print(f"ROC-AUC      : {roc:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))
